In [ ]:
# =========================
# TensorFlow GraphSAGE (NO PyG) for IBM Synthetic AML HI-Small
# End-to-end: mount drive -> load CSV -> build neighbor table -> train -> eval
# =========================

from google.colab import drive
drive.mount("/content/drive")

import numpy as np
import pandas as pd
import tensorflow as tf
import gc

# -------------------------
# 1) Load HI-Small CSV
# -------------------------
PATH = "/content/drive/MyDrive/HI-Small_Trans.csv"  # <-- change if needed
df = pd.read_csv(PATH)
print("df:", df.shape)
print(df.columns)

TIME_COL = "Timestamp"
SRC_ACC  = "Account"
DST_ACC  = "Account.1"
AMT_PAY  = "Amount Paid"
LBL_COL  = "Is Laundering"

df[LBL_COL] = df[LBL_COL].astype(np.int32)
df[AMT_PAY] = pd.to_numeric(df[AMT_PAY], errors="coerce").fillna(0.0).astype(np.float32)

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.sort_values(TIME_COL).reset_index(drop=True)

# -------------------------
# 2) Factorize nodes
# -------------------------
nodes = pd.Index(pd.concat([df[SRC_ACC].astype("string"), df[DST_ACC].astype("string")], ignore_index=True))
codes, uniques = pd.factorize(nodes, sort=False)

E = len(df)
src = codes[:E].astype(np.int32)
dst = codes[E:].astype(np.int32)
n_nodes = int(len(uniques))

y = df[LBL_COL].to_numpy(np.int32)
amt = np.log1p(df[AMT_PAY].to_numpy(np.float32))  # edge feature

del nodes, codes, uniques
gc.collect()

print("edges:", E, "nodes:", n_nodes, "pos_rate:", float(y.mean()))

# -------------------------
# 3) Build fixed-size neighbor table (undirected)
# -------------------------
K = 20
adj = [[] for _ in range(n_nodes)]
for u, v in zip(src, dst):
    adj[u].append(v)
    adj[v].append(u)

rng = np.random.default_rng(42)
neighbors = np.zeros((n_nodes, K), dtype=np.int32)

for i in range(n_nodes):
    nbrs = adj[i]
    if len(nbrs) == 0:
        neighbors[i] = i
    elif len(nbrs) >= K:
        neighbors[i] = rng.choice(nbrs, size=K, replace=False)
    else:
        neighbors[i] = rng.choice(nbrs, size=K, replace=True)

del adj
gc.collect()

print("neighbors:", neighbors.shape)

# -------------------------
# 4) Time split + TF datasets
# -------------------------
n = len(y)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)

train_ids = np.arange(0, n_train, dtype=np.int32)
val_ids   = np.arange(n_train, n_train + n_val, dtype=np.int32)
test_ids  = np.arange(n_train + n_val, n, dtype=np.int32)

BATCH = 65536

def make_ds(edge_ids, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(edge_ids)
    if shuffle:
        ds = ds.shuffle(200000, seed=42, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_ids, shuffle=True)
val_ds   = make_ds(val_ids, shuffle=False)
test_ds  = make_ds(test_ids, shuffle=False)

# Convert arrays to TF tensors once
src_tf = tf.constant(src)
dst_tf = tf.constant(dst)
amt_tf = tf.constant(amt)
y_tf   = tf.constant(y)
neighbors_tf = tf.constant(neighbors)

# -------------------------
# 5) TensorFlow GraphSAGE model
# -------------------------
class GraphSAGE(tf.keras.Model):
    def __init__(self, n_nodes, emb_dim=64, hidden=64, edge_hidden=128, dropout=0.2):
        super().__init__()
        self.emb = tf.keras.layers.Embedding(n_nodes, emb_dim)
        self.d1  = tf.keras.layers.Dense(hidden, activation="relu")
        self.d2  = tf.keras.layers.Dense(hidden, activation="relu")
        self.drop = tf.keras.layers.Dropout(dropout)

        self.edge_mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(edge_hidden, activation="relu"),
            tf.keras.layers.Dropout(dropout),
            tf.keras.layers.Dense(1)
        ])

    def sage_layer(self, h, nbr_idx):
        nbr_h = tf.gather(h, nbr_idx)              # [N, K, D]
        nbr_mean = tf.reduce_mean(nbr_h, axis=1)   # [N, D]
        return tf.concat([h, nbr_mean], axis=1)    # [N, 2D]

    def call(self, src_nodes, dst_nodes, edge_amt, neighbors, training=None):
        N = tf.shape(neighbors)[0]

        h0 = self.emb(tf.range(N))                 # [N, D]

        h1_in = self.sage_layer(h0, neighbors)
        h1 = self.d1(h1_in)
        h1 = self.drop(h1, training=training)

        h2_in = self.sage_layer(h1, neighbors)
        h2 = self.d2(h2_in)                        # [N, H]

        hs = tf.gather(h2, src_nodes)
        hd = tf.gather(h2, dst_nodes)

        edge_feat = tf.expand_dims(edge_amt, 1)
        z = tf.concat([hs, hd, edge_feat], axis=1)

        logits = self.edge_mlp(z, training=training)
        return tf.squeeze(logits, axis=1)

model = GraphSAGE(n_nodes, emb_dim=64, hidden=64, edge_hidden=128, dropout=0.2)
opt = tf.keras.optimizers.Adam(2e-3)

# class imbalance weighting
pos_rate = float(np.mean(y[train_ids]))
pos_weight = (1 - pos_rate) / max(pos_rate, 1e-9)
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True, reduction="none")
pos_weight_tf = tf.constant(pos_weight, dtype=tf.float32)

# -------------------------
# 6) Train/eval loops
# -------------------------
@tf.function
def train_step(edge_ids):
    s = tf.gather(src_tf, edge_ids)
    d = tf.gather(dst_tf, edge_ids)
    a = tf.gather(amt_tf, edge_ids)
    yy = tf.cast(tf.gather(y_tf, edge_ids), tf.float32)

    with tf.GradientTape() as tape:
        logits = model(s, d, a, neighbors_tf, training=True)
        loss_vec = bce(yy, logits)
        weights = tf.where(tf.equal(yy, 1.0), pos_weight_tf, 1.0)
        loss = tf.reduce_mean(loss_vec * weights)

    grads = tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def eval_ds(ds):
    ys, ps = [], []
    for edge_ids in ds:
        s = tf.gather(src_tf, edge_ids)
        d = tf.gather(dst_tf, edge_ids)
        a = tf.gather(amt_tf, edge_ids)
        logits = model(s, d, a, neighbors_tf, training=False)
        prob = tf.sigmoid(logits).numpy()
        ys.append(tf.gather(y_tf, edge_ids).numpy())
        ps.append(prob)

    y_true = np.concatenate(ys)
    y_prob = np.concatenate(ps)

    roc = tf.keras.metrics.AUC(curve="ROC")
    pr  = tf.keras.metrics.AUC(curve="PR")
    roc.update_state(y_true, y_prob)
    pr.update_state(y_true, y_prob)
    return float(roc.result().numpy()), float(pr.result().numpy())

EPOCHS = 20

for ep in range(1, EPOCHS + 1):
    losses = []
    for batch_ids in train_ds:
        losses.append(float(train_step(batch_ids).numpy()))
    val_roc, val_pr = eval_ds(val_ds)
    print(f"epoch {ep} | loss {np.mean(losses):.4f} | val ROC {val_roc:.4f} | val PR {val_pr:.4f}")

test_roc, test_pr = eval_ds(test_ds)
print("\nTEST ROC:", test_roc)
print("TEST PR :", test_pr)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
df: (5078345, 11)
Index(['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1',
       'Amount Received', 'Receiving Currency', 'Amount Paid',
       'Payment Currency', 'Payment Format', 'Is Laundering'],
      dtype='object')
edges: 5078345 nodes: 515080 pos_rate: 0.0010194266045335635
neighbors: (515080, 20)
epoch 1 | loss 0.1043 | val ROC 0.5023 | val PR 0.0010
epoch 2 | loss 0.0132 | val ROC 0.5848 | val PR 0.0029
epoch 3 | loss 0.0101 | val ROC 0.6035 | val PR 0.0043
epoch 4 | loss 0.0084 | val ROC 0.6097 | val PR 0.0064
epoch 5 | loss 0.0075 | val ROC 0.6068 | val PR 0.0077
epoch 6 | loss 0.0067 | val ROC 0.6039 | val PR 0.0095
epoch 7 | loss 0.0060 | val ROC 0.6000 | val PR 0.0116
epoch 8 | loss 0.0055 | val ROC 0.5985 | val PR 0.0190
epoch 9 | loss 0.0050 | val ROC 0.6001 | val PR 0.0272
epoch 10 | loss 0.0045 | val ROC 0.6021 | val PR 0.0318
e